In [ ]:
import sqlite3
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from category_encoders import OneHotEncoder
from IPython.display import VimeoVideo
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.utils.validation import check_is_fitted

warnings.simplefilter(action="ignore", category=FutureWarning)


In [ ]:
VimeoVideo("665414074", h="d441543f18", width=600)


In [ ]:
def wrangle(db_path):
    # Connect to database
    conn = sqlite3.connect(db_path)

    # Construct query
    query = """
        SELECT distinct(i.building_id) AS b_id,
           s.*,
           d.damage_grade
        FROM id_map AS i
        JOIN building_structure AS s ON i.building_id = s.building_id
        JOIN building_damage AS d ON i.building_id = d.building_id
        WHERE district_id = 4
    """
    # Read query results into DataFrame
    df = pd.read_sql(query, conn, index_col="b_id")

    # Identify leaky columns
    drop_cols = [col for col in df.columns if "post_eq" in col]
    
    # Create binary target
    df['damage_grade'] = df['damage_grade'].str[-1].astype(int)
    df['severe_damage'] = (df['damage_grade']>3).astype(int)

    # Drop old target
    drop_cols.append("damage_grade")

    # Drop multicollinearity column
    drop_cols.append("count_floors_pre_eq")

    # Drop high-cardinality categorical column
    drop_cols.append("building_id")

    # Drop columns
    df.drop(columns=drop_cols, inplace=True)

    
    return df


In [ ]:
VimeoVideo("665414541", h="dfe22afdfb", width=600)


In [ ]:
df = wrangle("../nepal.sqlite")
df.head()


In [ ]:
# Check your work
assert df.shape[0] == 70836, f"`df` should have 70,836 rows, not {df.shape[0]}."


In [ ]:
VimeoVideo("665414560", h="ad4bba19ed", width=600)


In [ ]:
print(df.info())


In [ ]:
# Check your work
assert (
    df.filter(regex="post_eq").shape[1] == 0
), "`df` still has leaky features. Try again!"


In [ ]:
VimeoVideo("665414603", h="12b3d2f23e", width=600)


In [ ]:
print(df["severe_damage"].value_counts())


In [ ]:
# Check your work
assert (
    "damage_grade" not in df.columns
), "Your DataFrame should not include the `'damage_grade'` column."
assert (
    "severe_damage" in df.columns
), "Your DataFrame is missing the `'severe_damage'` column."
assert (
    df["severe_damage"].value_counts().shape[0] == 2
), f"The `'damage_grade'` column should have only two unique values, not {df['severe_damage'].value_counts().shape[0]}"


In [ ]:
VimeoVideo("665414636", h="d34256b4e3", width=600)


In [ ]:
# Create correlation matrix
correlation = df.select_dtypes("number").drop(columns="severe_damage").corr()

# Plot heatmap of correlation
sns.heatmap(correlation);


In [ ]:
# Check your work
assert (
    "count_floors_pre_eq" not in df.columns
), "Did you drop the `'count_floors_pre_eq'` column?"


In [ ]:
VimeoVideo("665414667", h="f39c2c21bc", width=600)


In [ ]:
# Create boxplot
sns.boxplot(x="severe_damage", y="height_ft_pre_eq", data=df)

# Label axes
plt.xlabel("Severe Damage")
plt.ylabel("Height Pre-earthquake [ft.]")
plt.title("Distribution of Building Height by Class");


In [ ]:
VimeoVideo("665414684", h="81295d5bdb", width=600)


In [ ]:
# Plot value counts of "severe_damage"
df["severe_damage"].value_counts(normalize=True).plot(
    kind="bar", xlabel="Class", ylabel="Relative Frequency", title="Class Balance"
);


In [ ]:
VimeoVideo("665414697", h="ee2d4f28c6", width=600)


In [ ]:
majority_class_prop, minority_class_prop = df["severe_damage"].value_counts(normalize=True)
print(majority_class_prop, minority_class_prop)


In [ ]:
# Check your work
assert (
    majority_class_prop < 1
), "`majority_class_prop` should be a floating point number between 0 and 1."
assert (
    minority_class_prop < 1
), "`minority_class_prop` should be a floating point number between 0 and 1."


In [ ]:
VimeoVideo("665414718", h="6a1e0c1e53", width=600)


In [ ]:
# Create pivot table
foundation_pivot = pd.pivot_table(
    df, index="foundation_type", values="severe_damage", aggfunc=np.mean
                                 ).sort_values(by="severe_damage")

foundation_pivot


In [ ]:
VimeoVideo("665414734", h="46de83f96e", width=600)


In [ ]:
# Plot bar chart of `foundation_pivot`
foundation_pivot.plot(kind="barh", legend=None)

plt.axvline(majority_class_prop, linestyle="--", color="red", label="Majority Class")
plt.axvline(minority_class_prop, linestyle="--", color="green", label="Minority Class")
plt.legend(loc="lower right")


In [ ]:
VimeoVideo("665414748", h="8549a0f89c", width=600)


In [ ]:
# Check for high- and low-cardinality categorical features
df.select_dtypes('object').nunique()


In [ ]:
target = "severe_damage"
X = df.drop(columns=[target])
y = df[target]


In [ ]:
VimeoVideo("665414769", h="1bfddf07b2", width=600)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


In [ ]:
VimeoVideo("665414807", h="c997c58720", width=600)


In [ ]:
acc_baseline = y_train.value_counts(normalize=True).max()
print("Baseline Accuracy:", round(acc_baseline, 2))


In [ ]:
VimeoVideo("665414835", h="1d8673223e", width=600)


In [ ]:
# Build model
model = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    LogisticRegression(max_iter=1000)
    
)
# Fit model to training data
model.fit(X_train, y_train)


In [ ]:
# Check your work
assert isinstance(
    model, Pipeline
), f"`model` should be a Pipeline, not type {type(model)}."
assert isinstance(
    model[0], OneHotEncoder
), f"The first step in your Pipeline should be a OneHotEncoder, not type {type(model[0])}."
assert isinstance(
    model[-1], LogisticRegression
), f"The last step in your Pipeline should be LogisticRegression, not type {type(model[-1])}."
check_is_fitted(model)


In [ ]:
VimeoVideo("665414885", h="f35ff0e23e", width=600)


In [ ]:
acc_train = accuracy_score(y_train, model.predict(X_train))
acc_test = model.score(X_test, y_test )

print("Training Accuracy:", round(acc_train, 2))
print("Test Accuracy:", round(acc_test, 2))


In [ ]:
VimeoVideo("665414902", h="f9bdbe9e75", width=600)


In [ ]:
y_train_pred_proba = model.predict_proba(X_train)
print(y_train_pred_proba[:5])


In [ ]:
features = model.named_steps["onehotencoder"].get_feature_names()
importances = model.named_steps["logisticregression"].coef_[0]


In [ ]:
VimeoVideo("665414916", h="c0540604cd", width=600)


In [ ]:
odds_ratios = pd.Series(np.exp(importances), index=features).sort_values()
odds_ratios.head()


In [ ]:
VimeoVideo("665414943", h="56eb74d93e", width=600)


In [ ]:
# Horizontal bar chart, five largest coefficients
odds_ratios.tail().plot(kind="barh")
plt.xlabel("Odds Ratio")


In [ ]:
VimeoVideo("665414964", h="a61b881450", width=600)


In [ ]:
# Horizontal bar chart, five smallest coefficients
# Horizontal bar chart, five largest coefficients
odds_ratios.head().plot(kind="barh")
plt.xlabel("Odds Ratio")
